# Práctica — Clase 3. Procesos de Decisión de Markov (PDM)

Esta práctica te toca a ti. Trabajas de forma autónoma con el mismo `GridWorld` que construiste en clase, y resuelves dos ejercicios sobre el **Proceso de Decisión de Markov** (PDM) y la **ecuación de Bellman de expectación**.

**Cómo trabajar con este notebook:**
1. Ejecuta primero la sección "Preparación", que reconstruye el entorno de la clase
2. En cada ejercicio, completa las líneas marcadas con `# TODO`
3. Ejecuta tu solución y compara el resultado con lo que se describe en el enunciado
4. Solo después, despliega la solución oculta para comprobar tu respuesta

> **No mires la solución antes de intentarlo.** El valor de este ejercicio está en pelearte un poco con la ecuación de Bellman, no en leer la respuesta.

## Preparación: recupera el GridWorld de la clase

Estas celdas reproducen tal cual la clase `GridWorld` y las funciones `politica_aleatoria`, `evaluacion_iterativa_politica` e `imprimir_valores` que construiste en el notebook de teoría. Los ejercicios de hoy reutilizan estos mismos nombres, así que no los cambies.

In [ ]:
import numpy as np

ACCIONES = ["arriba", "abajo", "izquierda", "derecha"]

DESPLAZAMIENTOS = {
    "arriba": (-1, 0),
    "abajo": (1, 0),
    "izquierda": (0, -1),
    "derecha": (0, 1),
}

In [ ]:
class GridWorld:
    """
    Un mundo-cuadrícula de n_filas x n_columnas con una casilla terminal (la meta).

    - Los estados son tuplas (fila, columna).
    - Las acciones son las cuatro direcciones cardinales.
    - Las transiciones son deterministas: moverse contra un muro o el
      borde del tablero deja al agente en la misma casilla.
    - La recompensa es 'recompensa_paso' en cada movimiento (por defecto -1),
      salvo en la casilla terminal, donde el episodio ya ha terminado.
    """

    def __init__(self, n_filas=4, n_columnas=4, estados_terminales=None, recompensa_paso=-1.0):
        self.n_filas = n_filas
        self.n_columnas = n_columnas
        self.recompensa_paso = recompensa_paso

        # Por defecto, la meta es la esquina inferior derecha
        if estados_terminales is None:
            estados_terminales = [(n_filas - 1, n_columnas - 1)]
        self.estados_terminales = set(estados_terminales)

        # Lista de todos los estados (fila, columna)
        self.estados = [(f, c) for f in range(n_filas) for c in range(n_columnas)]
        self.acciones = ACCIONES

    def es_terminal(self, estado):
        return estado in self.estados_terminales

    def estado_siguiente(self, estado, accion):
        """Devuelve la casilla resultante de aplicar 'accion' desde 'estado'.
        Si el movimiento choca con el borde del tablero, el agente se queda quieto."""
        if self.es_terminal(estado):
            return estado

        df, dc = DESPLAZAMIENTOS[accion]
        fila, columna = estado
        nueva_fila = max(0, min(self.n_filas - 1, fila + df))
        nueva_columna = max(0, min(self.n_columnas - 1, columna + dc))
        return (nueva_fila, nueva_columna)

    def recompensa(self, estado, accion, estado_siguiente):
        """Recompensa inmediata de la transición. En este GridWorld es fija
        (-1 por paso), pero se recibe (s, a, s') completo para poder
        ampliar el ejemplo más adelante (por ejemplo, recompensas distintas por casilla)."""
        if self.es_terminal(estado):
            return 0.0
        return self.recompensa_paso


def politica_aleatoria(mundo):
    """Construye una política estocástica que, en cada estado, elige
    cualquiera de las cuatro acciones con la misma probabilidad (0.25).
    Se representa como un diccionario: politica[estado][accion] = probabilidad."""
    probabilidad_uniforme = 1.0 / len(mundo.acciones)
    return {
        estado: {accion: probabilidad_uniforme for accion in mundo.acciones}
        for estado in mundo.estados
    }


def evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """
    Calcula Vpi(s) para una política fija aplicando repetidamente
    la ecuación de Bellman de expectación:

        Vpi(s) = suma_a pi(a|s) * suma_s' P(s'|s,a) * [R(s,a,s') + gamma * Vpi(s')]

    Devuelve V como un diccionario {estado: valor} y el número de barridos usados.
    """
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)  # barrido síncrono: usamos siempre los valores del barrido anterior

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                valor_esperado += prob_accion * (r + gamma * V[estado_sig])

            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    return V, iteracion


def imprimir_valores(mundo, V):
    """Muestra V como una cuadrícula legible, para inspeccionar el resultado."""
    for f in range(mundo.n_filas):
        fila_textos = []
        for c in range(mundo.n_columnas):
            fila_textos.append(f"{V[(f, c)]:6.2f}")
        print(" ".join(fila_textos))

### Línea base: el GridWorld original

Ejecuta la celda siguiente para tener a mano la misma referencia que viste en clase: el GridWorld 4x4 con la meta en la esquina inferior derecha, su política aleatoria y su Vπ ya convergida. Vas a usar `mundo`, `politica` y `V` en el ejercicio 2, así que no los sobrescribas.

In [ ]:
mundo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0)
politica = politica_aleatoria(mundo)
V, num_barridos = evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4)

print(f"Convergió en {num_barridos} barridos.\n")
imprimir_valores(mundo, V)

---

## Ejercicio 1: cambia la disposición de recompensas y completa la evaluación

El `GridWorld` del capítulo coloca la meta siempre en la esquina inferior derecha, pero la clase acepta cualquier lista de `estados_terminales`. En este ejercicio vas a:

1. Construir un nuevo `GridWorld` con la meta en otro sitio —por ejemplo, en el centro del tablero, `[(1, 1)]`— o con **dos** metas a la vez —por ejemplo, `[(0, 3), (3, 0)]`—
2. Completar los huecos del bucle de evaluación iterativa de políticas para calcular Vπ sobre ese nuevo mundo

**Pistas:**
- El constructor es `GridWorld(n_filas=4, n_columnas=4, estados_terminales=..., recompensa_paso=-1.0)`
- Los tres huecos que tienes que rellenar dentro del bucle corresponden, en orden, a: el estado siguiente (`mundo.estado_siguiente`), la recompensa inmediata (`mundo.recompensa`) y el sumando de la ecuación de Bellman de expectación: `prob_accion * (r + gamma * V[estado_sig])`
- Si mueves la meta al centro, espera ver los valores más altos (menos negativos) alrededor de `(1, 1)` en vez de en la esquina
- Si usas dos metas, cada casilla "ve" el camino más corto hacia cualquiera de las dos

In [ ]:
# Paso 1: elige dónde van las metas de tu nuevo GridWorld
# Ejemplos: [(1, 1)]              -> una única meta en el centro
#           [(0, 3), (3, 0)]      -> dos metas en esquinas opuestas
nuevos_estados_terminales = None  # TODO: sustituye None por una lista de tuplas (fila, columna)

mundo_ejercicio = GridWorld(
    n_filas=4,
    n_columnas=4,
    estados_terminales=nuevos_estados_terminales,
    recompensa_paso=-1.0,
)

politica_ejercicio = politica_aleatoria(mundo_ejercicio)

In [ ]:
def evaluacion_iterativa_politica_ejercicio(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Misma evaluación iterativa de políticas que en clase, pero con
    tres huecos por completar dentro del bucle sobre las acciones."""
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]

                # TODO: calcula el estado siguiente de aplicar 'accion' desde 'estado'
                estado_sig = None

                # TODO: calcula la recompensa inmediata de esa transición (estado, accion, estado_sig)
                r = None

                # TODO: acumula en valor_esperado el sumando de Bellman de expectación:
                #       prob_accion * (r + gamma * V[estado_sig])
                valor_esperado += 0

            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    return V, iteracion


V_ejercicio, barridos_ejercicio = evaluacion_iterativa_politica_ejercicio(
    mundo_ejercicio, politica_ejercicio, gamma=0.9
)

print(f"Convergió en {barridos_ejercicio} barridos.\n")
imprimir_valores(mundo_ejercicio, V_ejercicio)

<details>
<summary>💡 Ver solución</summary>

```python
# Paso 1: elegimos la meta en el centro del tablero
nuevos_estados_terminales = [(1, 1)]

mundo_ejercicio = GridWorld(
    n_filas=4,
    n_columnas=4,
    estados_terminales=nuevos_estados_terminales,
    recompensa_paso=-1.0,
)

politica_ejercicio = politica_aleatoria(mundo_ejercicio)


def evaluacion_iterativa_politica_ejercicio(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]

                # Estado al que se llega tras aplicar la acción
                estado_sig = mundo.estado_siguiente(estado, accion)

                # Recompensa inmediata de esa transición
                r = mundo.recompensa(estado, accion, estado_sig)

                # Sumando de la ecuación de Bellman de expectación:
                # pi(a|s) * [R(s,a,s') + gamma * Vpi(s')]
                valor_esperado += prob_accion * (r + gamma * V[estado_sig])

            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    return V, iteracion


V_ejercicio, barridos_ejercicio = evaluacion_iterativa_politica_ejercicio(
    mundo_ejercicio, politica_ejercicio, gamma=0.9
)

print(f"Convergió en {barridos_ejercicio} barridos.\n")
imprimir_valores(mundo_ejercicio, V_ejercicio)

# Salida esperada aproximada (la meta ahora está en el centro, (1, 1)):
#  -5.82  -4.89  -6.59  -7.57
#  -4.89   0.00  -5.78  -7.48
#  -6.59  -5.78  -7.19  -7.97
#  -7.57  -7.48  -7.97  -8.34
#
# Fíjate en el patrón: el valor menos negativo (0.00) ya no está en la
# esquina, sino en (1, 1), y el resto de casillas se reorganiza alrededor
# de la nueva meta exactamente igual que antes lo hacía alrededor de la
# esquina inferior derecha.

# Variante con dos metas, para comprobar que el patrón se repite:
mundo_dos_metas = GridWorld(
    n_filas=4, n_columnas=4,
    estados_terminales=[(0, 3), (3, 0)],
    recompensa_paso=-1.0,
)
politica_dos_metas = politica_aleatoria(mundo_dos_metas)
V_dos_metas, barridos_dos_metas = evaluacion_iterativa_politica_ejercicio(
    mundo_dos_metas, politica_dos_metas, gamma=0.9
)
print(f"\nDos metas — convergió en {barridos_dos_metas} barridos.\n")
imprimir_valores(mundo_dos_metas, V_dos_metas)
```
</details>

---

## Ejercicio 2: la ecuación de Bellman de expectación, a mano

En este ejercicio no vas a usar `evaluacion_iterativa_politica`. Vas a calcular Vπ para **un único estado** aplicando la ecuación de Bellman de expectación directamente, sumando término a término sobre las cuatro acciones:

```
Vπ(s) = Σ_a π(a|s) · [R(s,a,s') + γ·Vπ(s')]
```

Usa el `mundo`, la `politica` y el `V` que ya calculaste en la sección "Línea base" de más arriba: para cada acción, necesitas `V[estado_sig]`, es decir, el valor que ya tiene el estado vecino según la evaluación completa. Al final, compara tu resultado con `V[estado_elegido]`: deberían coincidir, porque en un estado convergido la ecuación de Bellman ya se cumple.

**Pistas:**
- Recorre `mundo.acciones` con un bucle for
- Para cada acción, necesitas: la probabilidad `politica[estado_elegido][accion]`, el estado siguiente `mundo.estado_siguiente(estado_elegido, accion)`, la recompensa `mundo.recompensa(...)` y el valor `V[estado_sig]`
- Como `evaluacion_iterativa_politica` para en cuanto `delta_maximo < umbral`, los dos resultados serán prácticamente iguales, pero no bit a bit idénticos: la diferencia debería ser menor que el `umbral` (1e-4) usado para calcular `V`

In [ ]:
estado_elegido = (2, 3)  # TODO: puedes probar con cualquier estado no terminal del mundo
gamma = 0.9

valor_manual = 0.0

for accion in mundo.acciones:
    # TODO: obtén la probabilidad de esta acción en politica[estado_elegido]
    prob_accion = None

    # TODO: calcula el estado siguiente con mundo.estado_siguiente(...)
    estado_sig = None

    # TODO: calcula la recompensa inmediata con mundo.recompensa(...)
    r = None

    # TODO: acumula en valor_manual el sumando prob_accion * (r + gamma * V[estado_sig])
    valor_manual += 0

print(f"Valor calculado a mano para {estado_elegido}: {valor_manual:.4f}")
print(f"Valor de la función genérica para {estado_elegido}: {V[estado_elegido]:.4f}")
print(f"Diferencia: {abs(valor_manual - V[estado_elegido]):.6f}")

<details>
<summary>💡 Ver solución</summary>

```python
estado_elegido = (2, 3)
gamma = 0.9

valor_manual = 0.0

for accion in mundo.acciones:
    # Probabilidad que la política asigna a esta acción en el estado elegido
    prob_accion = politica[estado_elegido][accion]

    # Estado al que se llega tras aplicar la acción
    estado_sig = mundo.estado_siguiente(estado_elegido, accion)

    # Recompensa inmediata de esa transición
    r = mundo.recompensa(estado_elegido, accion, estado_sig)

    # Sumando de la ecuación de Bellman de expectación
    valor_manual += prob_accion * (r + gamma * V[estado_sig])

print(f"Valor calculado a mano para {estado_elegido}: {valor_manual:.4f}")
print(f"Valor de la función genérica para {estado_elegido}: {V[estado_elegido]:.4f}")
print(f"Diferencia: {abs(valor_manual - V[estado_elegido]):.6f}")

# Salida esperada aproximada:
# Valor calculado a mano para (2, 3): -5.7568
# Valor de la función genérica para (2, 3): -5.7568
# Diferencia: 0.0000

assert abs(valor_manual - V[estado_elegido]) < 1e-3, "Deberían coincidir, dentro del umbral de convergencia"
print("\nCoinciden: la ecuación de Bellman de expectación se cumple en el punto fijo Vpi.")
```

**Por qué funciona:** `evaluacion_iterativa_politica` no es más que aplicar esta misma cuenta a los dieciséis estados, una y otra vez, hasta que los valores dejan de moverse. Cuando ya convergió, aplicar la ecuación una vez más —a mano, para un solo estado— tiene que devolver (casi) el mismo número que ya estaba guardado en `V`. Esa es la definición de punto fijo: un estado en el que aplicar la ecuación de Bellman ya no cambia nada.
</details>

---

## Para seguir practicando

Si te ha sobrado tiempo, vuelve al `GridWorld` original y prueba estas variantes del capítulo:

1. **Cambia el factor de descuento.** Prueba con γ = 0.5 y con γ = 0.99. ¿Cómo cambian los valores de las casillas alejadas de la meta? ¿Qué le pasa al número de barridos que tarda en converger?
2. **Combina ambos ejercicios.** Repite el ejercicio 2 sobre el `mundo_ejercicio` del ejercicio 1, en vez de sobre el `mundo` original, y comprueba que la ecuación de Bellman se sigue cumpliendo

**Siguiente parada:** en la Clase 4 vas a reutilizar este mismo `GridWorld` para ir más allá de evaluar una política fija: vas a mejorarla hasta encontrar la política óptima π* mediante programación dinámica. ¡Guarda este notebook, lo necesitarás!